# Speculative Decoding 教程

本教程介绍推测解码技术，通过小模型加速大模型推理。

## 目录
1. 自回归解码的瓶颈
2. 推测解码原理
3. 拒绝采样
4. 实战演示

## 1. 自回归解码的瓶颈

LLM 推理是 memory-bound，每个 token 需要加载全部模型权重：

$$\text{Time} = N \times T_{\text{forward}}$$

其中 $N$ 是生成长度，$T_{\text{forward}}$ 是单次前向时间。

**问题**：GPU 计算能力未充分利用

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from src.speculative import (
    SpeculativeConfig,
    DraftModel,
    TokenVerifier,
    SpeculativeDecoder,
    create_speculative_decoder
)

# 计算理论加速比
def theoretical_speedup(k, acceptance_rate):
    """k: 推测长度, acceptance_rate: 接受率"""
    expected_tokens = sum(acceptance_rate**i for i in range(k+1))
    return expected_tokens  # 相比单 token 解码

print("理论加速比 (k=4):")
for rate in [0.7, 0.8, 0.9]:
    speedup = theoretical_speedup(4, rate)
    print(f"  接受率 {rate:.0%}: {speedup:.2f}x")

## 2. 推测解码原理

使用小模型 (Draft) 快速生成候选 token，大模型 (Target) 并行验证：

```
Draft Model:  [t1] → [t2] → [t3] → [t4]  (快速生成 k 个)
                ↓      ↓      ↓      ↓
Target Model: [验证 t1, t2, t3, t4]      (一次前向)
                ↓      ↓      ↓      ↓
Result:       [✓]    [✓]    [✗]    [-]  (接受前 2 个)
```

In [ ]:
# 创建推测解码配置
config = SpeculativeConfig(
    num_speculative_tokens=4,  # 每次推测 4 个 token
    draft_model_name="draft",
    target_model_name="target"
)

print(f"推测长度: {config.num_speculative_tokens}")
print(f"Draft 模型: {config.draft_model_name}")

## 3. 拒绝采样

使用拒绝采样保证输出分布与 Target 模型一致：

$$P(\text{accept}) = \min\left(1, \frac{p_{\text{target}}(x)}{p_{\text{draft}}(x)}\right)$$

如果拒绝，从修正分布采样：

$$p'(x) = \text{norm}\left(\max(0, p_{\text{target}}(x) - p_{\text{draft}}(x))\right)$$

In [ ]:
# 创建 Token 验证器
verifier = TokenVerifier()

# 模拟验证过程
draft_tokens = np.array([10, 20, 30, 40])  # Draft 生成的 token
draft_probs = np.array([0.8, 0.7, 0.6, 0.5])  # Draft 概率
target_probs = np.array([0.9, 0.6, 0.3, 0.4])  # Target 概率

# 计算接受概率
accept_probs = np.minimum(1.0, target_probs / draft_probs)
print("接受概率:", accept_probs)

## 4. 实战演示

In [ ]:
# 创建推测解码器
decoder = create_speculative_decoder(
    num_speculative_tokens=4,
    vocab_size=1000
)

print(f"推测解码器已创建")
print(f"词表大小: 1000")

In [ ]:
# 模拟生成
np.random.seed(42)

# 模拟 draft 和 target 的 logits
def mock_draft_forward(tokens):
    return np.random.randn(len(tokens), 1000)

def mock_target_forward(tokens):
    return np.random.randn(len(tokens), 1000)

# 统计接受率
total_draft = 0
total_accepted = 0

for _ in range(10):
    # 模拟一轮推测解码
    k = 4
    accepted = np.random.binomial(k, 0.75)  # 假设 75% 接受率
    total_draft += k
    total_accepted += accepted

print(f"平均接受率: {total_accepted/total_draft:.1%}")
print(f"理论加速: {(total_accepted + 10) / 10:.2f}x")

## 总结

推测解码的核心优势：

1. **无损加速**: 输出分布与原模型一致
2. **2-3x 加速**: 典型场景
3. **灵活配置**: 可调推测长度

### 参考资料
- [Speculative Decoding Paper](https://arxiv.org/abs/2211.17192)
- [Fast Inference from Transformers via Speculative Decoding](https://arxiv.org/abs/2302.01318)